In [1]:
# Cell 1: Cấu hình mô phỏng ⚙️
import torch
import os # For path joining

class SimulationConfig:
    # === 1. Cấu hình kiến trúc & Phân cấp ===
    ALGORITHM = 'hsfl_meanstd_balanced_agg' # Tên mới
    NUM_TIERS = 3
    ENTITIES_PER_TIER = [5, 1]
    CLIENT_MAPPING = 'balanced'
    GLOBAL_ROUNDS = 50
    
    # --- Cấu hình tham gia ---
    PARTICIPATION_RATE = 1.0 # 10% client
    MIN_PARTICIPANTS = 2
    
    # --- Cấu hình Aggregation ---
    CLIENT_AGG_INTERVAL = 5    # Aggregate Client models (Tier 1) mỗi 5 vòng
    EDGE_AGG_INTERVAL = 10     # Aggregate Edge models (Tier 2) mỗi 10 vòng

    # === 2. Cấu hình SimCLR & Huấn luyện Local ===
    SIMCLR_EPOCHS_CLIENT = 5
    SIMCLR_EPOCHS_EDGE = 10
    SIMCLR_EPOCHS_CLOUD = 20
    PROJECTION_DIM = 128
    f = 0.5
    REGULARIZATION_LAMBDA = 0.0 # <-- TẮT FEEDBACK

    # === 3. (SỬA) Cấu hình Prototype & Sample Generation ===
    PROTOTYPE_METHOD = 'mean_std' # <-- BẮT BUỘC
    NUM_PROTOTYPES = 50         # (Không dùng)
    SAMPLE_GENERATION_METHOD = 'gaussian_balanced' # <-- TÊN MỚI
    # (MỚI) Tổng số sample tạo ra ở mỗi tầng (Edge/Cloud)
    NUM_SYNTHETIC_SAMPLES_PER_TIER = 500 

    # === 4. Cấu hình Huấn luyện & Đánh giá ===
    BATCH_SIZE = 64
    LEARNING_RATE = 0.001
    LINEAR_PROBE_EPOCHS = 50

    TEMPERATURE = 1.0

    # === 5. Cấu hình hệ thống & đường dẫn ===
    CONFIG_PATH = os.path.join('..', 'configs', 'config.yaml')
    DATA_PATH = os.path.join('..', 'data')
    RESULTS_DIR = os.path.join('..', 'results', 'logs')
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    SEED = 42

config = SimulationConfig()

# --- Configuration Checks ---
assert len(config.ENTITIES_PER_TIER) == config.NUM_TIERS - 1
os.makedirs(config.RESULTS_DIR, exist_ok=True)
print(f"Algorithm: {config.ALGORITHM.upper()}")
print(f"Prototype Method: {config.PROTOTYPE_METHOD}")
print(f"Sample Generation: {config.SAMPLE_GENERATION_METHOD} ({config.NUM_SYNTHETIC_SAMPLES_PER_TIER} samples/tier)")
print(f"Device: {config.DEVICE}")

Algorithm: HSFL_MEANSTD_BALANCED_AGG
Prototype Method: mean_std
Sample Generation: gaussian_balanced (500 samples/tier)
Device: cuda


In [ ]:
# Cell 2: Tải cấu hình, dữ liệu và thiết lập cấu trúc phân cấp 📦
import yaml, torch, numpy as np, time, copy, os, collections, random
import matplotlib.pyplot as plt
from torch import nn
import torch.nn.functional as F

# --- Add src directory to path ---
import sys
sys.path.append(os.path.abspath(os.path.join('..')))
from src.dataset_preparation import prepare_data 
from src.optimizer import LARS

# --- Setup & Load Data Config ---
torch.manual_seed(config.SEED); np.random.seed(config.SEED); random.seed(config.SEED)
print(f"Reading data config from: {config.CONFIG_PATH}")
try:
    with open(config.CONFIG_PATH, 'r') as f: loaded_config = yaml.safe_load(f)
    data_params = loaded_config['data_config']
    print(f"Loaded data config: {data_params}")
except FileNotFoundError:
     print(f"FATAL: Config file not found at {config.CONFIG_PATH}. Run 00 script first.")
     raise
except Exception as e:
     print(f"FATAL: Error reading config file {config.CONFIG_PATH}: {e}")
     raise

# --- Prepare Data ---
try:
    client_loaders, test_loader, class_names = prepare_data(
        data_path=config.DATA_PATH, batch_size=config.BATCH_SIZE, seed=config.SEED, **data_params
    )
    total_num_clients = len(client_loaders); num_classes = len(class_names)
    print(f"\nInitialized data for {total_num_clients} clients.")
except Exception as e:
     print(f"FATAL: Error during data preparation: {e}")
     raise

# --- Setup Hierarchical Structure ---
entities_map = collections.defaultdict(list); client_parent_map = {}; entity_children_map = collections.defaultdict(list)
entities_map[0] = list(range(total_num_clients))
num_entities_tier2 = config.ENTITIES_PER_TIER[0]; entities_map[1] = list(range(num_entities_tier2))
client_indices_per_edge = np.array_split(np.arange(total_num_clients), num_entities_tier2)
for edge_id, client_indices in enumerate(client_indices_per_edge):
    entity_children_map[(1, edge_id)] = list(client_indices)
    for client_id in client_indices: client_parent_map[client_id] = (1, edge_id)
if config.NUM_TIERS > 2:
     num_entities_tier3 = config.ENTITIES_PER_TIER[1]; entities_map[2] = list(range(num_entities_tier3))
     entity_children_map[(2, 0)] = entities_map[1]

print("\nHierarchical Structure:"); print(f"  - Tier 1 (Clients): {total_num_clients}")
print(f"  - Tier 2 (Edges): {num_entities_tier2}")
if config.NUM_TIERS > 2: print(f"  - Tier 3 (Cloud): {num_entities_tier3}")

In [ ]:
# Cell 3: SimCLR Components 🧩
from torchvision import transforms
import torch.nn.functional as F

# --- Data Augmentation ---
def get_simclr_transforms(input_size=32):
    return transforms.Compose([
        transforms.ToPILImage(),
        transforms.RandomHorizontalFlip(), 
        transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
        transforms.RandomGrayscale(p=0.2), 
        transforms.ToTensor(),
    ])

# --- Projection Head ---
class ProjectionMLP(nn.Module):
    def __init__(self, in_features, hidden_dim=512, out_dim=128):
        super().__init__()
        if in_features <= 0: raise ValueError(f"ProjectionMLP input dim > 0, got {in_features}")
        self.mlp = nn.Sequential(nn.Linear(in_features, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, out_dim))
    def forward(self, x): return self.mlp(x)

# --- NT-Xent Loss ---
def nt_xent_loss(z_i, z_j, temperature):
    batch_size = z_i.shape[0];
    if batch_size == 0: return torch.tensor(0.0, device=z_i.device)
    z = torch.cat((z_i, z_j), dim=0); z = F.normalize(z, dim=1)
    sim_matrix = torch.mm(z, z.T) / temperature
    sim_matrix = sim_matrix - torch.eye(2 * batch_size, device=z.device) * 1e5
    labels = torch.arange(2 * batch_size, device=z.device); labels = (labels + batch_size) % (2 * batch_size)
    loss = F.cross_entropy(sim_matrix, labels)
    return loss

In [ ]:
# Cell 4: Model Definition & Initialization (Đã xóa VAE) 🧠
from torchvision.models import resnet18, ResNet18_Weights
import collections
import copy
from torch import nn
import torch

# --- XÓA Class VAE ---

# --- Hàm chia ResNet (Không thay đổi) ---
def create_hsfl_resnet_parts(num_tiers=3, input_sample_shape=(2, 3, 32, 32)):
    base_model = resnet18(weights=ResNet18_Weights.DEFAULT); modules = list(base_model.children())[:-1]
    model_parts = []; flattened_feature_dims = []; unflattened_shapes = []; dummy_input = torch.randn(*input_sample_shape)
    if num_tiers == 3:
        client_part = nn.Sequential(*modules[:5]); edge_part = nn.Sequential(*modules[5:7])
        cloud_part = nn.Sequential(*modules[7], nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(1))
        model_parts = [client_part, edge_part, cloud_part]
        with torch.no_grad():
            temp_client = client_part.to(config.DEVICE); temp_edge = edge_part.to(config.DEVICE); temp_cloud = cloud_part.to(config.DEVICE)
            dummy_input_device = dummy_input.to(config.DEVICE)
            out1 = temp_client(dummy_input_device); unflattened_shapes.append(out1.shape[1:]); flat_dim1 = nn.Flatten(1)(out1).shape[1]; flattened_feature_dims.append(flat_dim1)
            out2 = temp_edge(out1); unflattened_shapes.append(out2.shape[1:]); flat_dim2 = nn.Flatten(1)(out2).shape[1]; flattened_feature_dims.append(flat_dim2)
            out3 = temp_cloud(out2); unflattened_shapes.append(None); flat_dim3 = out3.shape[1]; flattened_feature_dims.append(flat_dim3)
            del temp_client, temp_edge, temp_cloud, dummy_input_device
            if config.DEVICE.type == 'cuda': torch.cuda.empty_cache()
        print(f"Flattened feature dims: {flattened_feature_dims}")
        print(f"Unflattened output shapes: {unflattened_shapes}")
    else: raise NotImplementedError("Only 3 tiers supported.")
    return model_parts, flattened_feature_dims, unflattened_shapes

# --- Tạo mô hình, projection heads VÀ LƯU SHAPES ---
input_size = 64 if data_params['dataset_name'] == 'tiny_imagenet' else 32
hsfl_model_parts, feature_dims, unflattened_output_shapes = create_hsfl_resnet_parts(
    config.NUM_TIERS, input_sample_shape=(2, 3, input_size, input_size)
)

# --- Khởi tạo instances (Xóa VAE) ---
models = collections.defaultdict(list); projections = collections.defaultdict(list); optimizers = collections.defaultdict(list)
# --- XÓA vaes, vae_optimizers ---

# Tier 1 (Clients)
print(f"Initializing {total_num_clients} models/projections/optimizers for Tier 1...")
for i in range(total_num_clients):
    model = copy.deepcopy(hsfl_model_parts[0]).to(config.DEVICE)
    try: proj = ProjectionMLP(feature_dims[0], out_dim=config.PROJECTION_DIM).to(config.DEVICE)
    except ValueError as e: print(f"Error client proj {i}: {e}"); raise e
    optimizer = LARS(list(model.parameters()) + list(proj.parameters()), lr=config.LEARNING_RATE)
    models[0].append(model); projections[0].append(proj); optimizers[0].append(optimizer)

# Tiers > 1 (Shared: Edges, Cloud)
for tier_idx in range(1, config.NUM_TIERS):
    num_entities = len(entities_map[tier_idx])
    output_feature_dim_h = feature_dims[tier_idx] # Input dim cho Proj Head
    print(f"Initializing {num_entities} models/projections/optimizers for Tier {tier_idx+1}...")
    for i in range(num_entities):
        # Main Model
        model = copy.deepcopy(hsfl_model_parts[tier_idx]).to(config.DEVICE)
        try: proj = ProjectionMLP(output_feature_dim_h, out_dim=config.PROJECTION_DIM).to(config.DEVICE)
        except ValueError as e: print(f"Error shared proj Tier {tier_idx+1}, Entity {i}: {e}"); raise e
        optimizer = LARS(list(model.parameters()) + list(proj.parameters()), lr=config.LEARNING_RATE)
        models[tier_idx].append(model); projections[tier_idx].append(proj); optimizers[tier_idx].append(optimizer)
        # --- VAE ĐÃ BỊ XÓA ---

# Get SimCLR transforms
simclr_transforms = get_simclr_transforms(input_size=input_size)

In [ ]:
# Cell 4: Model Definition & Initialization (Đã xóa VAE) 🧠
from torchvision.models import resnet18, ResNet18_Weights
import collections
import copy
from torch import nn
import torch

# --- XÓA Class VAE ---

# --- Hàm chia ResNet (Không thay đổi) ---
def create_hsfl_resnet_parts(num_tiers=3, input_sample_shape=(2, 3, 32, 32)):
    base_model = resnet18(weights=ResNet18_Weights.DEFAULT); modules = list(base_model.children())[:-1]
    model_parts = []; flattened_feature_dims = []; unflattened_shapes = []; dummy_input = torch.randn(*input_sample_shape)
    if num_tiers == 3:
        client_part = nn.Sequential(*modules[:5]); edge_part = nn.Sequential(*modules[5:7])
        cloud_part = nn.Sequential(*modules[7], nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(1))
        model_parts = [client_part, edge_part, cloud_part]
        with torch.no_grad():
            temp_client = client_part.to(config.DEVICE); temp_edge = edge_part.to(config.DEVICE); temp_cloud = cloud_part.to(config.DEVICE)
            dummy_input_device = dummy_input.to(config.DEVICE)
            out1 = temp_client(dummy_input_device); unflattened_shapes.append(out1.shape[1:]); flat_dim1 = nn.Flatten(1)(out1).shape[1]; flattened_feature_dims.append(flat_dim1)
            out2 = temp_edge(out1); unflattened_shapes.append(out2.shape[1:]); flat_dim2 = nn.Flatten(1)(out2).shape[1]; flattened_feature_dims.append(flat_dim2)
            out3 = temp_cloud(out2); unflattened_shapes.append(None); flat_dim3 = out3.shape[1]; flattened_feature_dims.append(flat_dim3)
            del temp_client, temp_edge, temp_cloud, dummy_input_device
            if config.DEVICE.type == 'cuda': torch.cuda.empty_cache()
        print(f"Flattened feature dims: {flattened_feature_dims}")
        print(f"Unflattened output shapes: {unflattened_shapes}")
    else: raise NotImplementedError("Only 3 tiers supported.")
    return model_parts, flattened_feature_dims, unflattened_shapes

# --- Tạo mô hình, projection heads VÀ LƯU SHAPES ---
input_size = 64 if data_params['dataset_name'] == 'tiny_imagenet' else 32
hsfl_model_parts, feature_dims, unflattened_output_shapes = create_hsfl_resnet_parts(
    config.NUM_TIERS, input_sample_shape=(2, 3, input_size, input_size)
)

# --- Khởi tạo instances (Xóa VAE) ---
models = collections.defaultdict(list); projections = collections.defaultdict(list); optimizers = collections.defaultdict(list)
# --- XÓA vaes, vae_optimizers ---

# Tier 1 (Clients)
print(f"Initializing {total_num_clients} models/projections/optimizers for Tier 1...")
for i in range(total_num_clients):
    model = copy.deepcopy(hsfl_model_parts[0]).to(config.DEVICE)
    try: proj = ProjectionMLP(feature_dims[0], out_dim=config.PROJECTION_DIM).to(config.DEVICE)
    except ValueError as e: print(f"Error client proj {i}: {e}"); raise e
    optimizer = LARS(list(model.parameters()) + list(proj.parameters()), lr=config.LEARNING_RATE)
    models[0].append(model); projections[0].append(proj); optimizers[0].append(optimizer)

# Tiers > 1 (Shared: Edges, Cloud)
for tier_idx in range(1, config.NUM_TIERS):
    num_entities = len(entities_map[tier_idx])
    output_feature_dim_h = feature_dims[tier_idx] # Input dim cho Proj Head
    print(f"Initializing {num_entities} models/projections/optimizers for Tier {tier_idx+1}...")
    for i in range(num_entities):
        # Main Model
        model = copy.deepcopy(hsfl_model_parts[tier_idx]).to(config.DEVICE)
        try: proj = ProjectionMLP(output_feature_dim_h, out_dim=config.PROJECTION_DIM).to(config.DEVICE)
        except ValueError as e: print(f"Error shared proj Tier {tier_idx+1}, Entity {i}: {e}"); raise e
        optimizer = LARS(list(model.parameters()) + list(proj.parameters()), lr=config.LEARNING_RATE)
        models[tier_idx].append(model); projections[tier_idx].append(proj); optimizers[tier_idx].append(optimizer)
        # --- VAE ĐÃ BỊ XÓA ---

# Get SimCLR transforms
simclr_transforms = get_simclr_transforms(input_size=input_size)

In [ ]:
# Cell 5: Helper Functions (Hybrid Version - Sửa lỗi encode_data) 🛠️
from sklearn.cluster import KMeans
from tqdm.notebook import tqdm # Sửa: dùng tqdm.notebook
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim
import numpy as np
import torch
from torchvision import transforms
from itertools import islice

# --- Helper to calculate tensor size ---
def calculate_tensor_size_mb(data_to_measure):
    """Calculates size of tensor or dict of tensors in MB."""
    total_size_mb = 0
    if data_to_measure is None: return 0
    elif isinstance(data_to_measure, torch.Tensor):
        total_size_mb = data_to_measure.element_size()*data_to_measure.nelement()/(1024**2)
    elif isinstance(data_to_measure, dict):
        for key, tensor in data_to_measure.items():
            if isinstance(tensor, torch.Tensor): total_size_mb += tensor.element_size()*tensor.nelement()/(1024**2)
    else: pass
    return total_size_mb

# --- (MỚI) Model Aggregation Function ---
def aggregate_models(weights_list):
    """Tính trung bình (FedAvg) một danh sách các state_dict."""
    if not weights_list:
        return None, 0.0
    start_time = time.time()
    
    aggregated_weights = collections.OrderedDict()
    valid_weights_list = [w for w in weights_list if w is not None]
    if not valid_weights_list: return None, 0.0
    
    for key in valid_weights_list[0].keys():
        aggregated_weights[key] = torch.zeros_like(valid_weights_list[0][key]).to(config.DEVICE)

    num_models = 0
    for state_dict in valid_weights_list:
         num_models += 1
         for key in aggregated_weights.keys():
              if key in state_dict:
                   aggregated_weights[key] += state_dict[key].to(config.DEVICE) 

    if num_models > 0:
        for key in aggregated_weights.keys():
            aggregated_weights[key] = torch.div(aggregated_weights[key], num_models)
    else:
        return None, 0.0
        
    aggregation_time = time.time() - start_time
    return aggregated_weights, aggregation_time

# --- VAE Loss Function ---
def vae_loss_function(recon_x, x, mu, logvar, beta=1.0):
    recon_loss = F.mse_loss(recon_x, x.view(-1, recon_x.shape[1]), reduction='sum') 
    kld_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kld_loss

# --- (SỬA LỖI) VAE Training Loop (fix lỗi BatchNorm) ---
def train_vae(vae_model, vae_optimizer, data_tensor, epochs, beta=1.0):
    """Huấn luyện VAE trên một tensor dữ liệu (ví dụ: các prototypes)."""
    if vae_model is None or vae_optimizer is None: return
    if data_tensor is None or data_tensor.numel() == 0:
        print("  Warn: No data provided to train VAE. Skipping."); return
        
    vae_model.train()
    dataset = TensorDataset(data_tensor)
    
    vae_batch_size = min(config.VAE_BATCH_SIZE, len(dataset))
    if vae_batch_size <= 1:
        print(f"    Warn: Not enough data ({len(dataset)} samples) for VAE batch > 1. Skipping VAE training.")
        vae_model.eval()
        return
        
    loader = DataLoader(dataset, batch_size=vae_batch_size, shuffle=True, drop_last=False)
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        num_batches_processed = 0
        for (batch_x,) in loader: 
            if batch_x.shape[0] <= 1: continue 
            vae_optimizer.zero_grad()
            recon_batch, mu, logvar = vae_model(batch_x)
            loss = vae_loss_function(recon_batch, batch_x, mu, logvar, beta)
            if not torch.isnan(loss) and not torch.isinf(loss):
                loss.backward()
                vae_optimizer.step()
                epoch_loss += loss.item()
                num_batches_processed += 1
    vae_model.eval()

# --- Prototype Calculation (Sửa KMeans n_init) ---
@torch.no_grad()
def compute_prototypes(model, projection_head, data_loader, method='kmeans', k=50, compute_on_z=False):
    if model is None and not compute_on_z: print("Error: Model None required for h-protos."); return None
    if compute_on_z and (model is None or projection_head is None): print("Error: Model & Proj needed for z-protos."); return None
    if model: model.eval()
    if projection_head and compute_on_z: projection_head.eval()
    all_features_flat_list = []; flattener = nn.Flatten(1).to(config.DEVICE)
    data_iterator = data_loader if isinstance(data_loader, DataLoader) else data_loader
    for i, batch_data in enumerate(data_iterator):
        if isinstance(batch_data, (list, tuple)): data = batch_data[0].to(config.DEVICE)
        else: data = batch_data.to(config.DEVICE)
        try:
            if model is None: continue
            features_h = model(data); features_h_flat = flattener(features_h)
            if compute_on_z:
                if projection_head is None: continue
                if features_h_flat.shape[1] != projection_head.mlp[0].in_features: continue
                features_to_use = projection_head(features_h_flat)
            else: features_to_use = features_h_flat
            all_features_flat_list.append(features_to_use.detach().cpu().numpy())
        except Exception as e: print(f"  Error proto feature process: {e}. Input: {data.shape}. Skip batch."); continue
    if not all_features_flat_list: print("Warn: No features collected."); return None
    try: all_features_flat = np.concatenate(all_features_flat_list, axis=0)
    except ValueError as e: print(f"Error concat features: {e}."); return None
    num_samples = all_features_flat.shape[0]
    if num_samples < k and method == 'kmeans': k = max(1, num_samples)
    if method == 'kmeans':
        if k <= 0: return None;
        if num_samples == 0: return None
        kmeans = KMeans(n_clusters=k, random_state=config.SEED, n_init='auto', verbose=0) # Sửa: n_init='auto'
        try: kmeans.fit(all_features_flat); return torch.tensor(kmeans.cluster_centers_, dtype=torch.float32)
        except Exception: return None
    elif method == 'mean_std':
        if num_samples == 0: return None
        mean = np.mean(all_features_flat, axis=0); std = np.std(all_features_flat, axis=0)
        if mean.shape != std.shape: std = np.zeros_like(mean)
        return {'mean': torch.tensor(mean, dtype=torch.float32), 'std': torch.tensor(std, dtype=torch.float32)}
    else: raise ValueError("Invalid proto method.")

# --- SimCLR Training Loop (Đã xóa Feedback) ---
def simclr_train_loop(model, projection, optimizer, data_source, epochs, is_real_data=True,
                      expected_input_shape=None, parent_projected_data=None, reg_lambda=0.0):
    """Inner training loop for SimCLR (No feedback)."""
    model.train(); projection.train(); losses = []; flattener = nn.Flatten(1).to(config.DEVICE)
    for epoch in range(epochs):
        epoch_loss = 0.0; num_batches = 0
        if is_real_data: data_iterator = tqdm(data_source, desc=f"SimCLR Ep {epoch+1}/{epochs}", leave=False); num_batches = len(data_source)
        else:
            if data_source is None or data_source.numel() == 0: print(f"  Skip Ep {epoch+1}: No data."); continue
            pseudo_dataset = TensorDataset(data_source); pseudo_loader = DataLoader(pseudo_dataset, batch_size=config.BATCH_SIZE, shuffle=True)
            data_iterator = tqdm(pseudo_loader, desc=f"SimCLR Ep {epoch+1}/{epochs}", leave=False); num_batches = len(pseudo_loader)
        if num_batches == 0: print(f"  Skip Ep {epoch+1}: Empty loader."); continue

        for i, batch in enumerate(data_iterator):
            optimizer.zero_grad()
            total_loss = torch.tensor(0.0).to(config.DEVICE)
            try:
                if is_real_data:
                    images, _ = batch; img1_list, img2_list = [], []
                    for img_idx, img in enumerate(images):
                         try: img1_list.append(simclr_transforms(img)); img2_list.append(simclr_transforms(img))
                         except Exception as ex: print(f"Warn: Error transforms: {ex}. Skip img."); continue
                    if not img1_list or not img2_list or len(img1_list) != len(img2_list): continue
                    img1 = torch.stack(img1_list).to(config.DEVICE); img2 = torch.stack(img2_list).to(config.DEVICE)
                else:
                    features = batch[0].to(config.DEVICE); current_batch_size = features.shape[0]
                    if expected_input_shape is None: raise ValueError("expected_input_shape required")
                    try: reshaped_features = features.view(current_batch_size, *expected_input_shape)
                    except RuntimeError as e: print(f"  Error reshaping: {e}. Skip batch."); continue
                    noise1 = torch.randn_like(reshaped_features)*0.01; noise2 = torch.randn_like(reshaped_features)*0.01
                    img1 = reshaped_features + noise1; img2 = reshaped_features + noise2
                
                h1 = model(img1); h2 = model(img2);
                if h1.dim() > 2: h1_flat = flattener(h1); h2_flat = flattener(h2)
                else: h1_flat, h2_flat = h1, h2
                z1 = projection(h1_flat); z2 = projection(h2_flat)
                loss_simclr = nt_xent_loss(z1, z2, config.TEMPERATURE); total_loss = loss_simclr

                if parent_projected_data is not None and reg_lambda > 0:
                     # (Feedback đã bị tắt, nhưng giữ logic để không lỗi)
                     pass 

                if not torch.isnan(total_loss) and not torch.isinf(total_loss) and total_loss.requires_grad:
                     total_loss.backward(); optimizer.step()
                     epoch_loss += total_loss.item(); data_iterator.set_postfix(loss=total_loss.item())
                else: print(f"  Warn: Invalid loss ({total_loss.item()}) or no grad. Skipping step.")
            except Exception as e: print(f"  Error SimCLR batch {i}: {e}. Skip."); continue
        avg_loss = epoch_loss / max(num_batches, 1); losses.append(avg_loss)
    return losses

def generate_samples(distributions_list, num_samples_per_dist, method='gaussian_balanced'):
    # ...
    if method == 'gaussian_balanced':
        if not isinstance(distributions_list, list): ... # Kiểm tra
            
        all_samples = []
        valid_dist_count = 0
        
        # (A) Lặp qua TỪNG dict {'mean':..., 'std':...} nhận được
        for i, stats_dict in enumerate(distributions_list):
            # ... (Kiểm tra dict hợp lệ) ...
            mean = stats_dict['mean']
            std = stats_dict['std']
            
            # (B) Tạo ra CHÍNH XÁC 'num_samples_per_dist' MẪU
            #     từ CHỈ mean/std này
            n_samples_this = num_samples_per_dist 
            samples = torch.randn(n_samples_this, len(mean), device=mean.device) \
                      * std.clamp(min=1e-6) + mean
            
            all_samples.append(samples)
            valid_dist_count += 1
            
        # (C) Gộp tất cả các sample cân bằng lại
        return torch.cat(all_samples, dim=0)
# --- (SỬA LỖI) Linear Probing Evaluation ---
@torch.no_grad()
def encode_data(all_models, all_projections, data_loader, client_id=0):
    """Encodes data through the entire model chain, ending with the FINAL projection head."""
    eval_model_chain = []; valid_path = True
    
    # Xây dựng chuỗi model path
    try:
        eval_model_chain.append(all_models[0][client_id]) # Client model
        for tier_idx in range(1, config.NUM_TIERS):
             entity_id = get_entity_id(tier_idx, client_id) # Lấy Edge/Cloud ID
             eval_model_chain.append(all_models[tier_idx][entity_id]) # Edge, Cloud models
        
        # Thêm projection head cuối cùng (của Cloud)
        final_tier_idx = config.NUM_TIERS - 1
        final_entity_id = get_entity_id(final_tier_idx, 0) # client_id=0 -> cloud 0
        eval_model_chain.append(all_projections[final_tier_idx][final_entity_id])
    except Exception as e:
         print(f"Error building eval chain: {e}"); valid_path = False
            
    if not valid_path: return None, None
    
    for m in eval_model_chain:
        if m is not None: m.eval() # Thêm kiểm tra None
        
    all_features, all_labels = [], []
    # --- (XÓA) flattener KHÔNG CẦN THIẾT ---
    data_iterator = data_loader if isinstance(data_loader, DataLoader) else data_loader

    for data_batch in tqdm(data_iterator, desc="Encoding data", leave=False):
        labels=None
        if isinstance(data_batch,(list,tuple)): data=data_batch[0].to(config.DEVICE); labels=data_batch[1] if len(data_batch)>1 else None
        else: data=data_batch.to(config.DEVICE)
        
        features = data # Bắt đầu với ảnh 4D: [B, 3, 32, 32]
        try:
            # --- (SỬA LỖI) Xóa logic flatten thủ công ---
            for i, model_part in enumerate(eval_model_chain):
                 if model_part is None: raise ValueError(f"Model part {i} is None in eval chain.")
                 # print(f"  Eval step {i} ({type(model_part).__name__}), input shape: {features.shape}") # Debug
                 features = model_part(features)
                 # Client model (i=0) ra 4D [B, 64, 8, 8]
                 # Edge model (i=1) ra 4D [B, 256, 4, 4]
                 # Cloud model (i=2) ra 2D [B, 512] (vì có flatten bên trong)
                 # Cloud proj (i=3) ra 2D [B, 128]
            # --- KẾT THÚC SỬA LỖI ---
                 
            all_features.append(features.detach().cpu())
            if labels is not None: all_labels.append(labels.cpu())
        except Exception as e: print(f"  Error encoding batch: {e}. Input shape: {data.shape}. Skip."); continue
        
    if not all_features: return None, None
    try: final_features = torch.cat(all_features); final_labels = torch.cat(all_labels) if all_labels and all_features else None
    except RuntimeError as e: print(f"Error concat encoded: {e}"); return None, None
    return final_features, final_labels

def evaluate_linear_probe(train_features, train_labels, test_features, test_labels):
    """Trains and evaluates a linear classifier on extracted features ('z' space)."""
    if train_features is None or train_labels is None or test_features is None or test_labels is None: print("Linear Probe skipped: Missing data."); return 0.0
    if len(train_features) == 0 or len(test_features) == 0: print("Linear Probe skipped: Empty features."); return 0.0
    if len(train_features) != len(train_labels): print(f"Linear Probe skipped: Mismatched train ({len(train_features)} vs {len(train_labels)})."); return 0.0
    if len(test_features) != len(test_labels): print(f"Linear Probe skipped: Mismatched test ({len(test_features)} vs {len(test_labels)})."); return 0.0

    input_dim = train_features.shape[1]; global num_classes
    linear_clf = nn.Linear(input_dim, num_classes).to(config.DEVICE); clf_optimizer = LARS(linear_clf.parameters(), lr=0.01); clf_criterion = nn.CrossEntropyLoss()
    train_dataset = TensorDataset(train_features, train_labels); probe_batch_size = min(512, len(train_dataset))
    if probe_batch_size == 0: print("Linear Probe skipped: Train dataset empty."); return 0.0
    train_loader = DataLoader(train_dataset, batch_size=probe_batch_size, shuffle=True)

    print("Training Linear Probe..."); linear_clf.train()
    for epoch in range(config.LINEAR_PROBE_EPOCHS):
        for features, labels in train_loader:
            features, labels = features.to(config.DEVICE), labels.to(config.DEVICE); clf_optimizer.zero_grad()
            logits = linear_clf(features); loss = clf_criterion(logits, labels); loss.backward(); clf_optimizer.step()
    linear_clf.eval(); correct = 0; total = 0
    test_dataset = TensorDataset(test_features, test_labels); test_loader_probe = DataLoader(test_dataset, batch_size=512)
    with torch.no_grad():
         for features, labels in test_loader_probe:
              features, labels = features.to(config.DEVICE), labels.to(config.DEVICE)
              logits = linear_clf(features); preds = logits.argmax(dim=1); correct += (preds == labels).sum().item(); total += labels.size(0)
    accuracy = 100. * correct / total if total > 0 else 0.0; print(f"Linear Probe Test Accuracy: {accuracy:.2f}%"); return accuracy
    
# --- get_entity_id function ---
def get_entity_id(tier_idx, client_id):
    if tier_idx == 0: return client_id
    elif tier_idx == 1:
        if client_id in client_parent_map:
            parent_tier, parent_id = client_parent_map[client_id];
            if parent_tier == 1: return parent_id
            else: print(f"Warn: Client {client_id} parent not Tier 2."); return 0
        else: print(f"Warn: Client {client_id} not in map."); return 0
    else:
        num_entities_this_tier = len(entities_map.get(tier_idx, []))
        if num_entities_this_tier >= 1: return 0
        else: print(f"Error: No entities Tier {tier_idx+1}."); return None

In [ ]:
# Cell 6: Vòng lặp huấn luyện chính (HSFL + MeanStd + Balanced Sampling + Hybrid Agg) 🚀
# (SỬA ĐỔI: Cloud Tier dùng Cross-Entropy thay vì Contrastive)
import collections
import time
import random
import torch
import numpy as np
from tqdm import tqdm 
from torch.utils.data import DataLoader, TensorDataset
from itertools import islice
import pandas as pd
import os
import torch.nn as nn

# Initialize history dictionary
history = {'rounds': [], 'accuracy': [], 'comm_cost': [], 'comp_cost': []}
print(f"\n--- BẮT ĐẦU MÔ PHỎNG: {config.ALGORITHM.upper()} ---")

# --- (Feedback đã bị tắt) ---

# --- Global Training Loop ---
for current_round in range(1, config.GLOBAL_ROUNDS + 1):
    if config.DEVICE.type == 'cuda': torch.cuda.empty_cache()
    round_start_time = time.time()
    print(f"\n--- Global Round {current_round}/{config.GLOBAL_ROUNDS} ---")
    comm_cost_round = 0.0; round_comp_cost = 0.0
    current_upward_protos_h = collections.defaultdict(list)

    # --- Chọn 10% Client tham gia ---
    num_to_select = max(config.MIN_PARTICIPANTS, int(total_num_clients * config.PARTICIPATION_RATE))
    selected_client_ids = random.sample(range(total_num_clients), num_to_select)
    print(f"  Chọn {num_to_select}/{total_num_clients} client tham gia vòng này.")

    # --- Tier 1: Client Training & Proto Calc (NO Feedback) ---
    print("  Tầng 1 (Client): Huấn luyện SimCLR...")
    client_train_start_time = time.time()
    client_outputs_buffer = []
    
    for client_id in tqdm(selected_client_ids, desc="Client Training", leave=False):
        model = models[0][client_id]; proj = projections[0][client_id]; optimizer = optimizers[0][client_id]
        loader = client_loaders[client_id]
        simclr_train_loop(model, proj, optimizer, loader, config.SIMCLR_EPOCHS_CLIENT, is_real_data=True)
        # Compute 'h' protos (mean_std) to send UP
        proto_h = compute_prototypes(model, None, loader, config.PROTOTYPE_METHOD, k=config.NUM_PROTOTYPES, compute_on_z=False)
        if proto_h is not None:
            parent_tier, parent_id = client_parent_map.get(client_id, (None, None))
            if parent_id is not None: client_outputs_buffer.append({'parent_id': parent_id, 'data': proto_h})
    round_comp_cost += (time.time() - client_train_start_time)

    # --- Communication: Client -> Edge (Send 'h' mean_std dicts) ---
    print("  Giao tiếp: Client -> Edge...")
    edge_received_data_h = collections.defaultdict(list) # {edge_id: [dict1, dict2, ...]}
    for output in client_outputs_buffer:
        parent_id = output['parent_id']; data = output['data']
        edge_received_data_h[parent_id].append(data); comm_cost_round += calculate_tensor_size_mb(data)
    del client_outputs_buffer
    if config.DEVICE.type == 'cuda': torch.cuda.empty_cache()

    # --- Tier 2: Edge Sample Gen & SimCLR Training ---
    print(f"  Tầng 2 (Edge): Sample Generation & SimCLR Training...")
    current_edge_upward_h = []
    edge_train_start_time = time.time()
    active_edge_ids = [edge_id for edge_id in entities_map[1] if edge_received_data_h[edge_id]]
    print(f"  {len(active_edge_ids)}/{len(entities_map[1])} Edges đang hoạt động.")
    
    for edge_id in tqdm(active_edge_ids, desc="Edge Training", leave=False):
          received_list_h = edge_received_data_h[edge_id] # Đây là list các dict mean_std
          if not received_list_h: print(f"  Warn: No valid data for Edge {edge_id}. Skip."); continue

          # --- (SỬA) Generate Samples (Balanced) ---
          num_sources = len(received_list_h) # Số lượng client con đã gửi
          if num_sources == 0: continue
          # Tính số sample CÂN BẰNG cho mỗi client con
          num_samples_per_source = max(1, config.NUM_SYNTHETIC_SAMPLES_PER_TIER // num_sources) 
          
          gen_features = generate_samples(
                received_list_h, # Truyền list các dicts
                num_samples_per_source, # Số sample TỪ MỖI dict
                config.SAMPLE_GENERATION_METHOD
          )
          if gen_features is None: print(f"  Warn: Sample gen failed Edge {edge_id}. Skip."); continue
          gen_features = gen_features.to(config.DEVICE)
          # --- KẾT THÚC SỬA ---

          # --- Train Edge SimCLR ---
          model = models[1][edge_id]; proj = projections[1][edge_id]; optimizer = optimizers[1][edge_id]
          expected_shape_for_edge_input = unflattened_output_shapes[0]
          simclr_train_loop(
                model, proj, optimizer, gen_features, config.SIMCLR_EPOCHS_EDGE, is_real_data=False,
                expected_input_shape=expected_shape_for_edge_input
          )

          # --- Compute Edge 'h' Prototypes (dùng sample vừa tạo) ---
          reshaped_gen_features_for_proto = None
          try: reshaped_gen_features_for_proto = gen_features.view(gen_features.shape[0], *expected_shape_for_edge_input)
          except RuntimeError as e: print(f"  Error reshaping Edge {edge_id} proto: {e}.")
          if reshaped_gen_features_for_proto is not None:
                pseudo_ds = TensorDataset(reshaped_gen_features_for_proto); pseudo_loader = DataLoader(pseudo_ds, batch_size=config.BATCH_SIZE)
                proto_h = compute_prototypes(model, None, pseudo_loader, config.PROTOTYPE_METHOD, k=config.NUM_PROTOTYPES, compute_on_z=False)
                if proto_h is not None: current_edge_upward_h.append({'parent_id': 0, 'data': proto_h})
          
          del gen_features, reshaped_gen_features_for_proto
          if 'pseudo_ds' in locals(): del pseudo_ds, pseudo_loader
    round_comp_cost += (time.time() - edge_train_start_time)
    
    del edge_received_data_h
    # (SỬA) Xóa an toàn
    if 'data_for_sampling' in locals(): del data_for_sampling
    if config.DEVICE.type == 'cuda': torch.cuda.empty_cache()

    # --- Communication: Edge -> Cloud ---
    if config.NUM_TIERS > 2:
        print("  Communication: Edge -> Cloud...")
        cloud_received_data_h = [] # Cloud receives 'h' mean_std dicts from Edges
        for output in current_edge_upward_h:
              data = output['data'];
              if data is not None: cloud_received_data_h.append(data); comm_cost_round += calculate_tensor_size_mb(data)
        del current_edge_upward_h

        # --- Tier 3: Cloud Sample Gen & (SỬA) Classification Training ---
        print(f"  Tier 3 (Cloud): Sample Generation & Classification Training...")
        cloud_train_start_time = time.time()
        if cloud_received_data_h:
              # --- Generate Samples (Balanced) ---
              num_sources_cloud = len(cloud_received_data_h) # Số lượng edge đã gửi
              if num_sources_cloud > 0:
                    # Tính số sample CÂN BẰNG cho mỗi edge
                    num_samples_per_source_cloud = max(1, config.NUM_SYNTHETIC_SAMPLES_PER_TIER // num_sources_cloud)
                    
                    gen_features_cloud = generate_samples(
                          cloud_received_data_h, # Truyền list các dicts
                          num_samples_per_source_cloud, # Số sample TỪ MỖI dict
                          config.SAMPLE_GENERATION_METHOD
                    )
                    
                    if gen_features_cloud is not None:
                        gen_features_cloud = gen_features_cloud.to(config.DEVICE)
                        cloud_model = models[2][0] # Lấy encoder
                        
                        # --- (BẮT ĐẦU SỬA) ---
                        # Thay vì SimCLR, chúng ta dùng Cross-Entropy
                        
                        # 1. Tạo pseudo-labels
                        #    Nhiệm vụ: Phân loại xem sample đến từ edge source nào
                        #    Tạo labels [0, 0, ..., 1, 1, ..., N-1, N-1]
                        #    trong đó N = num_sources_cloud
                        labels_list = []
                        for i in range(num_sources_cloud):
                            labels_list.extend([i] * num_samples_per_source_cloud)
                        
                        gen_labels_cloud = torch.tensor(labels_list, dtype=torch.long).to(config.DEVICE)
                        
                        # Đảm bảo số lượng sample = số lượng label
                        if gen_labels_cloud.shape[0] != gen_features_cloud.shape[0]:
                            print(f"    Warn: Label/Feature mismatch: {gen_labels_cloud.shape[0]} vs {gen_features_cloud.shape[0]}. Truncating.")
                            min_len = min(gen_labels_cloud.shape[0], gen_features_cloud.shape[0])
                            gen_labels_cloud = gen_labels_cloud[:min_len]
                            gen_features_cloud = gen_features_cloud[:min_len]

                        if gen_labels_cloud.shape[0] == 0:
                            print("    Warn: No data generated for Cloud CE training. Skipping.")
                            continue # Bỏ qua nếu không có data

                        # 2. Chuẩn bị model và data
                        expected_shape_for_cloud_input = unflattened_output_shapes[1]
                        cloud_model.train() # Đặt encoder ở chế độ train
                        
                        # 3. Tạo Classifier Head động (vì num_sources_cloud có thể thay đổi)
                        #    Lấy feature dim bằng cách chạy 1 sample
                        feature_dim = -1
                        cloud_model.eval() # Đặt tạm thời sang eval để tránh dropout/bn
                        try:
                            with torch.no_grad():
                                dummy_input = gen_features_cloud[0:1].view(1, *expected_shape_for_cloud_input)
                                feature_dim = cloud_model(dummy_input).shape[1]
                        except Exception as e:
                            print(f"    Error getting feature dim: {e}. Skipping cloud train.")
                            continue
                        cloud_model.train() # Quay lại chế độ train
                        num_cloud_classes = 10
                        cloud_classifier = nn.Linear(feature_dim, num_cloud_classes).to(config.DEVICE)
                        
                        # 4. Tạo Optimizer (cho cả encoder và classifier mới)
                        #    Chúng ta fine-tune encoder (cloud_model) VÀ huấn luyện classifier
                        optimizer = LARS(
                            list(cloud_model.parameters()) + list(cloud_classifier.parameters()), 
                            lr=config.LEARNING_RATE
                        )
                        criterion = nn.CrossEntropyLoss()
                        
                        # 5. Tạo DataLoader cho data tổng hợp
                        synthetic_dataset = TensorDataset(gen_features_cloud, gen_labels_cloud)
                        synthetic_loader = DataLoader(synthetic_dataset, batch_size=config.BATCH_SIZE, shuffle=True)

                        # 6. Vòng lặp huấn luyện Cross-Entropy
                        print(f"    Training Cloud CE (Classes={num_cloud_classes}, Epochs={config.SIMCLR_EPOCHS_CLOUD})...")
                        for epoch in range(config.SIMCLR_EPOCHS_CLOUD): # Tái sử dụng config epochs
                            for features, labels in synthetic_loader:
                                # Reshape features
                                try:
                                    features = features.view(features.shape[0], *expected_shape_for_cloud_input)
                                except RuntimeError as e:
                                    print(f"    Warn: Skipping batch, reshape error: {e}")
                                    continue
                                
                                optimizer.zero_grad()
                                
                                # Forward pass
                                h = cloud_model(features) # Lấy features từ encoder
                                logits = cloud_classifier(h) # Lấy logits từ head
                                
                                # Calculate loss
                                loss = criterion(logits, labels)
                                
                                # Backward pass
                                loss.backward()
                                optimizer.step()
                        
                        # 7. Dọn dẹp
                        del gen_labels_cloud, synthetic_dataset, synthetic_loader
                        del cloud_classifier, optimizer, criterion
                        
                        # --- (KẾT THÚC SỬA) ---
                        
                    else: print("  Warn: Sample gen failed Cloud.")
              else: print("  Warn: No valid distributions received by Cloud.")
        else: print("  Skipping Cloud training: No data received.")
        round_comp_cost += (time.time() - cloud_train_start_time)
        del cloud_received_data_h
        if 'gen_features_cloud' in locals(): del gen_features_cloud
        if config.DEVICE.type == 'cuda': torch.cuda.empty_cache()

    # --- Model Weight Aggregation ---
    # 1. Aggregate Client Models
    if config.CLIENT_AGG_INTERVAL > 0 and current_round % config.CLIENT_AGG_INTERVAL == 0:
        print(f"    * Aggregating Tier 1 (Client) models (Round {current_round})...")
        agg_start_time = time.time()
        client_models_to_agg = collections.defaultdict(list)
        client_projs_to_agg = collections.defaultdict(list)
        for cid in selected_client_ids:
            parent_tier, parent_id = client_parent_map.get(cid, (None, None))
            if parent_id is not None:
                client_models_to_agg[parent_id].append(models[0][cid].state_dict())
                client_projs_to_agg[parent_id].append(projections[0][cid].state_dict())
                for param in models[0][cid].state_dict().values(): comm_cost_round += calculate_tensor_size_mb(param)
                for param in projections[0][cid].state_dict().values(): comm_cost_round += calculate_tensor_size_mb(param)
        for edge_id, model_state_dicts in client_models_to_agg.items():
            proj_state_dicts = client_projs_to_agg.get(edge_id, [])
            if not model_state_dicts or not proj_state_dicts: continue
            aggregated_model_state, agg_cost_m = aggregate_models(model_state_dicts)
            aggregated_proj_state, agg_cost_p = aggregate_models(proj_state_dicts)
            round_comp_cost += (agg_cost_m + agg_cost_p)
            if aggregated_model_state and aggregated_proj_state:
                client_ids_in_group = entity_children_map.get((1, edge_id), [])
                for cid in client_ids_in_group:
                    models[0][cid].load_state_dict(aggregated_model_state)
                    projections[0][cid].load_state_dict(aggregated_proj_state)
                    optimizers[0][cid] = LARS(list(models[0][cid].parameters()) + list(projections[0][cid].parameters()), lr=config.LEARNING_RATE)
                    for param in aggregated_model_state.values(): comm_cost_round += calculate_tensor_size_mb(param)
                    for param in aggregated_proj_state.values(): comm_cost_round += calculate_tensor_size_mb(param)
        round_comp_cost += (time.time() - agg_start_time)
        print(f"    * Client aggregation complete.")

    # 2. Aggregate Edge Models (KHÔNG CÓ VAE)
    if config.EDGE_AGG_INTERVAL > 0 and current_round % config.EDGE_AGG_INTERVAL == 0:
        if config.NUM_TIERS > 2 and len(models[1]) > 1:
            print(f"    * Aggregating Tier 2 (Edge) models (Round {current_round})...")
            agg_start_time = time.time()
            models_to_agg = [models[1][eid].state_dict() for eid in entities_map[1]]
            projs_to_agg = [projections[1][eid].state_dict() for eid in entities_map[1]]
            
            if models_to_agg and projs_to_agg:
                for state_dict_list in [models_to_agg, projs_to_agg]:
                    for state_dict in state_dict_list:
                        for param in state_dict.values(): comm_cost_round += calculate_tensor_size_mb(param)
                aggregated_edge_state, agg_cost_m = aggregate_models(models_to_agg)
                aggregated_proj_state, agg_cost_p = aggregate_models(projs_to_agg)
                round_comp_cost += (agg_cost_m + agg_cost_p)
                
                if aggregated_edge_state and aggregated_proj_state:
                    for i in range(len(models[1])):
                        models[1][i].load_state_dict(aggregated_edge_state)
                        projections[1][i].load_state_dict(aggregated_proj_state)
                        optimizers[1][i] = LARS(list(models[1][i].parameters()) + list(projections[1][i].parameters()), lr=config.LEARNING_RATE)
                        for state_dict in [aggregated_edge_state, aggregated_proj_state]:
                            for param in state_dict.values(): comm_cost_round += calculate_tensor_size_mb(param)
            round_comp_cost += (time.time() - agg_start_time)
            print(f"    * Edge aggregation complete.")
            
    # --- Evaluation (Linear Probing) ---
    print("  Evaluation (Linear Probing)...")
    eval_start_time = time.time()
    accuracy = 0.0
    probe_train_loader = None;
    try:
        _ = client_loaders[0].dataset[0]; ds_len = len(client_loaders[0].dataset)
        subset_indices = list(range(ds_len))[:min(10000, ds_len)]
        if subset_indices:
              subset_sampler = torch.utils.data.SubsetRandomSampler(subset_indices)
              probe_train_loader = DataLoader(client_loaders[0].dataset, batch_size=config.BATCH_SIZE, sampler=subset_sampler)
        else: print("Warn: Client 0 dataset empty?")
    except TypeError:
        print("  Warn: Using first batches probe train (IterableDataset).")
        probe_train_loader = list(islice(client_loaders[0], max(1, 10000 // config.BATCH_SIZE)))
        if not probe_train_loader: print("Warn: Could not get probe batches.")

    if probe_train_loader:
          train_features, train_labels = encode_data(models, projections, probe_train_loader, client_id=0)
          test_features, test_labels = encode_data(models, projections, test_loader, client_id=0)
          if train_features is not None and train_labels is not None and test_features is not None and test_labels is not None:
                accuracy = evaluate_linear_probe(train_features, train_labels, test_features, test_labels)
          else: print("Skipping evaluation: Feature encoding failed.")
    else: print("Skipping evaluation: Could not create probe loader.")
    round_comp_cost += (time.time() - eval_start_time)
    
    if 'probe_train_loader' in locals(): del probe_train_loader
    if 'train_features' in locals(): del train_features, train_labels, test_features, test_labels
    if config.DEVICE.type == 'cuda': torch.cuda.empty_cache()

    # --- Log History ---
    round_elapsed_time = time.time() - round_start_time
    history['rounds'].append(current_round); history['accuracy'].append(accuracy)
    history['comm_cost'].append(comm_cost_round); history['comp_cost'].append(round_comp_cost)

    print(f"--- Global Round {current_round:02d} Summary ---")
    print(f"  Accuracy (Linear Probe): {accuracy:.2f}%")
    print(f"  Est. Comp Cost: {round_comp_cost:.2f} s")
    print(f"  Est. Comm Cost: {comm_cost_round:.2f} MB")
    print(f"  Round Time: {round_elapsed_time:.2f} s")

print("\n--- SIMULATION COMPLETE ---")

# --- Save Results ---
try:
    history_df = pd.DataFrame.from_dict(history)
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    algo_name = config.ALGORITHM
    dataset_name = data_params.get('dataset_name', 'unknown_ds')
    dist_mode = data_params.get('distribution_mode', 'unknown_dist')
    num_clients_actual = data_params.get('num_clients', total_num_clients)
    part_rate = config.PARTICIPATION_RATE
    client_agg = config.CLIENT_AGG_INTERVAL; edge_agg = config.EDGE_AGG_INTERVAL

    filename = f"{algo_name}_ds-{dataset_name}_dist-{dist_mode}_clients-{num_clients_actual}-p{part_rate}_agg-{client_agg}-{edge_agg}_samples-{config.NUM_SYNTHETIC_SAMPLES_PER_TIER}_{timestamp}.csv"
    save_path = os.path.join(config.RESULTS_DIR, filename)
    history_df.to_csv(save_path, index=False)
    print(f"\n✅ Simulation history saved to: {os.path.abspath(save_path)}")
except NameError: print("\nWarning: Could not save history, pandas (pd) not imported.")
except Exception as e: print(f"\n❌ Error saving history: {e}")

  Giao tiếp: Client -> Edge...
  Tầng 2 (Edge): Sample Generation & SimCLR Training...
  5/5 Edges đang hoạt động.


  Communication: Edge -> Cloud...
  Tier 3 (Cloud): Sample Generation & Classification Training...
    Training Cloud CE (Classes=10, Epochs=20)...
  Evaluation (Linear Probing)...


Training Linear Probe...
Linear Probe Test Accuracy: 12.16%
--- Global Round 01 Summary ---
  Accuracy (Linear Probe): 12.16%
  Est. Comp Cost: 761.79 s
  Est. Comm Cost: 0.66 MB
  Round Time: 761.79 s

--- Global Round 2/50 ---
  Chọn 20/20 client tham gia vòng này.
  Tầng 1 (Client): Huấn luyện SimCLR...


  Giao tiếp: Client -> Edge...
  Tầng 2 (Edge): Sample Generation & SimCLR Training...
  5/5 Edges đang hoạt động.


  Communication: Edge -> Cloud...
  Tier 3 (Cloud): Sample Generation & Classification Training...
    Training Cloud CE (Classes=10, Epochs=20)...
  Evaluation (Linear Probing)...


Training Linear Probe...
Linear Probe Test Accuracy: 10.67%
--- Global Round 02 Summary ---
  Accuracy (Linear Probe): 10.67%
  Est. Comp Cost: 698.05 s
  Est. Comm Cost: 0.66 MB
  Round Time: 698.05 s

--- Global Round 3/50 ---
  Chọn 20/20 client tham gia vòng này.
  Tầng 1 (Client): Huấn luyện SimCLR...


  Giao tiếp: Client -> Edge...
  Tầng 2 (Edge): Sample Generation & SimCLR Training...
  5/5 Edges đang hoạt động.


  Error SimCLR batch 296: CUDA error: unspecified launch failure
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
. Skip.


RuntimeError: CUDA error: unspecified launch failure
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# Cell 7: Plot Results 📊
import matplotlib.pyplot as plt
import pandas as pd # Import pandas if not already imported

if 'history' in locals() and history.get('rounds'):
    fig, ax_acc = plt.subplots(1, 1, figsize=(10, 6))
    title = (f"Algorithm: {config.ALGORITHM.upper()} ({config.NUM_TIERS} Tiers) | Dataset: {data_params.get('dataset_name','?')} "
             f"({data_params.get('distribution_mode','?')}) | Clients: {total_num_clients} (P={config.PARTICIPATION_RATE})\n"
             f"Proto: {config.PROTOTYPE_METHOD} | SampleGen: {config.SAMPLE_GENERATION_METHOD} ({config.NUM_SYNTHETIC_SAMPLES_PER_TIER}) | Agg: C-{config.CLIENT_AGG_INTERVAL} E-{config.EDGE_AGG_INTERVAL}")
    fig.suptitle(title, fontsize=12)
    rounds = history['rounds']
    color_acc = 'tab:blue'
    ax_acc.plot(rounds, history['accuracy'], marker='o', color=color_acc, label='Accuracy (Linear Probe)')
    ax_acc.set_xlabel("Global Round")
    ax_acc.set_ylabel("Test Accuracy (%)", color=color_acc)
    ax_acc.tick_params(axis='y', labelcolor=color_acc)
    ax_acc.grid(True); ax_acc.set_ylim(bottom=0)
    ax_comm = ax_acc.twinx()
    color_comm = 'tab:purple'
    ax_comm.plot(rounds, history['comm_cost'], marker='d', color=color_comm, linestyle='--', label='Est. Comm Cost')
    ax_comm.set_ylabel('Est. Comm Cost per Round (MB)', color=color_comm)
    ax_comm.tick_params(axis='y', labelcolor=color_comm)
    ax_comm.set_ylim(bottom=0)
    lines_acc, labels_acc = ax_acc.get_legend_handles_labels()
    lines_comm, labels_comm = ax_comm.get_legend_handles_labels()
    ax_comm.legend(lines_acc + lines_comm, labels_acc + labels_comm, loc='center right')
    plt.tight_layout(rect=[0, 0, 1, 0.90])
    plt.show()
else:
    print("History dictionary not found or empty. Skipping plot.")